In [6]:

import os
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import fitz  # PyMuPDF
from pathlib import Path

In [ ]:

# ------------------------------------------------------------
# PURE TEXT EXTRACTION (NO CLEANING, NO FILTERS)
# ------------------------------------------------------------

def extract_text_fitz_raw(pdf_path, txt_path):
    try:
        full_text = []

        with fitz.open(pdf_path) as doc:
            for page in doc:
                # Get raw text exactly as PyMuPDF extracts it
                text = page.get_text("text")
                full_text.append(text)

        # Join pages exactly with double newline
        final_text = "\n\n".join(full_text)

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(final_text)

    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")

In [8]:
# ------------------------------------------------------------
# INPUT YEARS
# ------------------------------------------------------------

START = int(input("Enter Start Year: "))
END   = int(input("Enter End Year: "))

BASE_DIR = f"D:/LPA_MTech_Project/My_Datasets/SC_{START}-{END}"
OUT_DIR  = f"D:/LPA_MTech_Project/Extracted_Texts/Extracts_{START}-{END}"

os.makedirs(OUT_DIR, exist_ok=True)

In [9]:

# ------------------------------------------------------------
# COLLECT PDFs
# ------------------------------------------------------------

def collect_pdfs(base_dir, start, end):
    pdfs = []
    for yr in range(start, end + 1):
        year_path = os.path.join(base_dir, str(yr), "english")
        if os.path.exists(year_path):
            found = [
                os.path.join(year_path, f)
                for f in os.listdir(year_path)
                if f.lower().endswith(".pdf")
            ]
            pdfs.extend(found)
            print(f"[{yr}] Found {len(found)} PDFs")
    print("TOTAL:", len(pdfs))
    return pdfs


pdf_files = collect_pdfs(BASE_DIR, START, END)

[2010] Found 907 PDFs
[2011] Found 848 PDFs
[2012] Found 623 PDFs
[2013] Found 853 PDFs
[2014] Found 795 PDFs
[2015] Found 760 PDFs
[2016] Found 589 PDFs
[2017] Found 732 PDFs
[2018] Found 795 PDFs
[2019] Found 1050 PDFs
TOTAL: 7952


In [10]:
# ------------------------------------------------------------
# MULTI-THREAD EXTRACTION
# ------------------------------------------------------------

with ThreadPoolExecutor(max_workers=6) as ex:
    for p in tqdm(pdf_files, desc="Extracting PDFs"):
        base = Path(p).stem
        txt_path = os.path.join(OUT_DIR, base + ".txt")
        ex.submit(extract_text_fitz_raw, p, txt_path)

print("DONE — Raw text saved to:", OUT_DIR)

Extracting PDFs: 100%|██████████| 7952/7952 [00:14<00:00, 555.08it/s] 


DONE — Raw text saved to: D:/LPA_MTech_Project/Extracted_Texts/Extracts_2010-2019
